# Matched-field processing

Matched-field processing (MFP) extends plane-wave beamforming to three
dimensions by replacing the plane-wave steering vector with a
*replica* — a predicted wavefield for a source at each candidate grid
point.  This allows simultaneous estimation of back-azimuth, horizontal
range, and depth.

This example uses the `covseisnet.matchedfield` module to locate a
synthetic point source embedded in a constant-velocity model, using the
*UnderVolc* array at Piton de la Fournaise. We demonstrate:

1. Building a constant-velocity model over the array area,
2. Computing per-station travel-time grids with
   `covseisnet.travel_times.TravelTimes`,
3. Applying the **Bartlett** and **MVDR** matched-field processors,
4. Comparing horizontal and vertical slices of the resulting power maps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import covseisnet as csn
from covseisnet.matchedfield import MatchedFieldProcessing
import covseisnet.synthetic as syn

## Station geometry

We load the UnderVolc seismograms, attach coordinates, and define a
geographical grid centred on the array.

In [ ]:
# Load waveforms and attach station coordinates
stream = csn.read("../data/undervolc.mseed")
stream.assign_coordinates("../data/undervolc.xml")

stats = [tr.stats for tr in stream]
n_stations = len(stats)

lons = np.array([s.coordinates["longitude"] for s in stats])
lats = np.array([s.coordinates["latitude"] for s in stats])
print(f"Array: {n_stations} stations")
print(f"Lon: {lons.min():.3f} – {lons.max():.3f}°")
print(f"Lat: {lats.min():.3f} – {lats.max():.3f}°")

## Velocity model and travel times

We build a constant-velocity model at 2.5 km/s over a grid that fully
contains the array, extended to 10 km depth. For each station we
compute the travel time from every grid point to that receiver.

In [ ]:
# Grid extent: 0.3° margin around the array, 0–10 km depth
margin = 0.1
extent = (
    lons.min() - margin, lons.max() + margin,
    lats.min() - margin, lats.max() + margin,
    0.0, 10.0,
)

velocity_model = csn.velocity.VelocityModel(
    extent=extent, shape=(30, 30, 15), velocity=2.5
)

# Per-station travel-time grids
travel_times = {
    tr.stats.station: csn.calculate_travel_times(
        velocity_model, stats=tr.stats
    )
    for tr in stream
}
print(f"TravelTimes grid shape: {next(iter(travel_times.values())).shape}")

## Validation on a synthetic spherical wave

We first verify the MFP implementation by injecting a synthetic
rank-1 covariance from a known point source and checking that the
Bartlett processor recovers the correct location.

In [ ]:
# True source: centre of the array, 3 km depth
source = (np.mean(lons), np.mean(lats), 3.0)
frequency = 2.0   # Hz
slowness  = 1 / 2.5  # s/km (matching the velocity model)

# Synthetic spherical-wave covariance
cov_synth = syn.spherical_wave_covariance(stats, frequency, slowness, source)

# Bartlett MFP
mfp = MatchedFieldProcessing(travel_times)
mfp.compute_bartlett(cov_synth, frequency)

lon_det, lat_det, dep_det = mfp.maximum_coordinates()
print(f"True source  : lon={source[0]:.3f}° lat={source[1]:.3f}° depth={source[2]:.1f} km")
print(f"MFP estimate : lon={lon_det:.3f}° lat={lat_det:.3f}° depth={dep_det:.1f} km")

In [ ]:
# Horizontal and vertical slices through the maximum
i_lon = np.argmin(np.abs(mfp.lon - lon_det))
i_lat = np.argmin(np.abs(mfp.lat - lat_det))
i_dep = np.argmin(np.abs(mfp.depth - dep_det))

LON, LAT, DEP = mfp.mesh

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# -- Horizontal slice at the detected depth
ax = axes[0]
pcm = ax.pcolormesh(
    LON[:, :, i_dep], LAT[:, :, i_dep], mfp[:, :, i_dep],
    cmap="inferno", shading="auto",
)
ax.scatter(lons, lats, marker="^", color="cyan", zorder=5, label="Stations")
ax.scatter(*source[:2], marker="*", s=200, color="lime", zorder=6, label="True source")
ax.scatter(lon_det, lat_det, marker="x", s=200, c="red", zorder=6, label="MFP estimate")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.set_title(f"Bartlett MFP – horizontal slice (depth {mfp.depth[i_dep]:.1f} km)")
ax.legend(fontsize=8)
plt.colorbar(pcm, ax=ax, label="Power")

# -- Vertical (lon–depth) slice at the detected latitude
ax = axes[1]
pcm = ax.pcolormesh(
    LON[:, i_lat, :], DEP[:, i_lat, :], mfp[:, i_lat, :],
    cmap="inferno", shading="auto",
)
ax.scatter(*[source[0], source[2]], marker="*", s=200, color="lime", zorder=6)
ax.scatter(lon_det, dep_det, marker="x", s=200, c="red", zorder=6)
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Depth (km)")
ax.invert_yaxis()
ax.set_title(f"Bartlett MFP – vertical slice (lat {mfp.lat[i_lat]:.3f}°)")
plt.colorbar(pcm, ax=ax, label="Power")

plt.tight_layout()

## MVDR matched-field processor

The MVDR (minimum-variance distortionless-response) processor typically
achieves higher spatial resolution than Bartlett by suppressing the
sidelobe structure at the cost of a matrix inversion.  We apply it to
the same synthetic covariance and compare the horizontal slice with the
Bartlett result.

In [ ]:
mfp_mvdr = MatchedFieldProcessing(travel_times)
mfp_mvdr.compute_mvdr(cov_synth, frequency)

lon_mv, lat_mv, dep_mv = mfp_mvdr.maximum_coordinates()
print(f"MVDR estimate: lon={lon_mv:.3f}° lat={lat_mv:.3f}° depth={dep_mv:.1f} km")

# Compare horizontal slices
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, result, title, sym in zip(
    axes,
    [mfp, mfp_mvdr],
    ["Bartlett", "MVDR"],
    ["x", "+"],
):
    i_d = np.argmin(np.abs(result.depth - result.maximum_coordinates()[2]))
    pcm = ax.pcolormesh(
        LON[:, :, i_d], LAT[:, :, i_d], result[:, :, i_d],
        cmap="inferno", shading="auto",
    )
    ax.scatter(lons, lats, marker="^", color="cyan", s=40, zorder=5)
    ax.scatter(*source[:2], marker="*", s=200, color="lime", zorder=6)
    ax.scatter(
        *result.maximum_coordinates()[:2],
        marker=sym, s=200, c="red", zorder=6,
    )
    ax.set_xlabel("Longitude (°)")
    ax.set_ylabel("Latitude (°)")
    ax.set_title(f"{title} MFP (depth {result.depth[i_d]:.1f} km)")
    plt.colorbar(pcm, ax=ax, label="Power")

plt.tight_layout()

## Application to real ambient noise

Finally, we estimate the covariance matrix from the real seismograms
and apply the Bartlett MFP at 2 Hz to image the dominant noise source.

In [ ]:
# Pre-process
stream.detrend("linear")
stream.filter("bandpass", freqmin=0.5, freqmax=8.0)

# Covariance matrix
times, frequencies, covariances = csn.calculate_covariance_matrix(
    stream, window_duration=20, average=10, whiten="slice"
)

# Pick the covariance closest to the target frequency
f_target = 2.0
i_f = np.argmin(np.abs(frequencies - f_target))
cov_real = covariances.mean(axis=0)[i_f]
print(f"Using frequency {frequencies[i_f]:.3f} Hz")

In [ ]:
mfp_real = MatchedFieldProcessing(travel_times)
mfp_real.compute_bartlett(cov_real, frequencies[i_f])

lon_r, lat_r, dep_r = mfp_real.maximum_coordinates()
print(f"Dominant source at {lon_r:.3f}° {lat_r:.3f}° depth {dep_r:.1f} km")

LON_r, LAT_r, DEP_r = mfp_real.mesh
i_d_r = np.argmin(np.abs(mfp_real.depth - dep_r))

fig, ax = plt.subplots(figsize=(6, 5))
pcm = ax.pcolormesh(
    LON_r[:, :, i_d_r], LAT_r[:, :, i_d_r], mfp_real[:, :, i_d_r],
    cmap="inferno", shading="auto",
)
ax.scatter(lons, lats, marker="^", color="cyan", zorder=5, label="Stations")
ax.scatter(lon_r, lat_r, marker="*", s=250, color="lime", zorder=6, label="MFP max")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.set_title(
    f"Bartlett MFP – real data at {frequencies[i_f]:.2f} Hz\n"
    f"(horizontal slice, depth {mfp_real.depth[i_d_r]:.1f} km)"
)
ax.legend(fontsize=8)
plt.colorbar(pcm, ax=ax, label="Power")
plt.tight_layout()